# Vitara AI — Vision Training Pipeline

> **Dataset Contract Ref:** Dataset Vision — Food Images  
> **Target Output:** `models/vision_model/`, `models/vision_model.tflite`  
> **Runtime:** Google Colab T4 GPU


## 1. Setup & Dependencies


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

print(f"TensorFlow version: {tf.__version__}")
if tf.config.list_physical_devices('GPU'):
    print("GPU is available!")
else:
    print("GPU is NOT available.")


## 2. Mount Google Drive & Config


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_ROOT   = "/content/drive/MyDrive"
PROCESSED_DIR = os.path.join(GDRIVE_ROOT, "data/vision/processed")
LOG_DIR       = os.path.join(GDRIVE_ROOT, "logs/vision")
MODELS_DIR    = os.path.join(GDRIVE_ROOT, "models")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
MAX_CALORIES = 1000.0  # Constant used for normalization


## 3. Unzip Dataset
Mengekstrak dataset gambar yang sudah di-split (Train, Val, Test) dari file zip.


In [ ]:
import zipfile
import os

ZIP_PATH = os.path.join(GDRIVE_ROOT, "data/vision/processed/dataset.zip")
EXTRACT_DIR = "/content/dataset"

print(f"Unzipping {ZIP_PATH} to {EXTRACT_DIR}...")
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)
print("Unzip complete!")


## 4. Data Augmentation & Generators
Menggunakan `ImageDataGenerator` untuk augmentasi data latih dan preprocessing (resize, normalisasi MobileNetV2).


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import pandas as pd

train_dir = os.path.join(EXTRACT_DIR, 'train')
val_dir = os.path.join(EXTRACT_DIR, 'val')
test_dir = os.path.join(EXTRACT_DIR, 'test')

# ImageDataGenerator dengan augmentasi untuk training
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# ImageDataGenerator HANYA preprocessing untuk validasi & testing
val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    shuffle=False
)

num_classes = len(train_generator.class_indices)
print(f"Number of classes: {num_classes}")

# --- Multi-Output Wrapper ---
# Membaca map kalori dari dataset mentah
calorie_df = pd.read_csv(os.path.join(GDRIVE_ROOT, "data/vision/raw/calorie_map.csv"))
class_to_calorie = dict(zip(calorie_df['class_name'], calorie_df['calories_per_100g']))

# Mapping dari index class generator ke nilai kalori (dinormalisasi)
idx_to_calorie = {v: class_to_calorie[k] / MAX_CALORIES for k, v in train_generator.class_indices.items()}

class MultiOutputSequence(keras.utils.Sequence):
    def __init__(self, generator, idx_to_calorie):
        self.generator = generator
        self.idx_to_calorie = idx_to_calorie
        
    def __len__(self):
        return len(self.generator)
        
    def __getitem__(self, index):
        x, y = self.generator[index]
        calories = np.array([self.idx_to_calorie[int(label)] for label in y])
        return x, {'classification_head': y, 'calorie_head': calories}
    
    def on_epoch_end(self):
        self.generator.on_epoch_end()

train_ds = MultiOutputSequence(train_generator, idx_to_calorie)
val_ds = MultiOutputSequence(val_generator, idx_to_calorie)
test_ds = MultiOutputSequence(test_generator, idx_to_calorie)


## 5. Build Model Architecture (Functional API)
`MobileNetV2 (pretrained, freeze awal) → GlobalAveragePooling2D → Dense → [classification_head (softmax), calorie_head (linear)]`


In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze awal

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
shared_dense = layers.Dense(128, activation='relu')(x)

classification_head = layers.Dense(num_classes, activation='softmax', name='classification_head')(shared_dense)
calorie_head = layers.Dense(1, activation='linear', name='calorie_head')(shared_dense)

model = Model(inputs=inputs, outputs=[classification_head, calorie_head])
model.summary()


## 6. Compile Model


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        'classification_head': 'sparse_categorical_crossentropy',
        'calorie_head': 'mae'
    },
    metrics={
        'classification_head': 'accuracy',
        'calorie_head': 'mae'
    },
    loss_weights={
        'classification_head': 1.0,
        'calorie_head': 1.0
    }
)

callbacks = [
    keras.callbacks.TensorBoard(log_dir=LOG_DIR),
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(filepath=os.path.join(MODELS_DIR, 'vision_model_best.keras'), save_best_only=True)
]


## 7. Initial Training


In [ ]:
EPOCHS = 20
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)


## 8. Fine-Tuning
Unfreeze layer atas MobileNetV2 setelah epoch awal konvergen.


In [ ]:
base_model.trainable = True
# Freeze semua kecuali 20 layer teratas
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5), # Lower learning rate
    loss={
        'classification_head': 'sparse_categorical_crossentropy',
        'calorie_head': 'mae'
    },
    metrics={
        'classification_head': 'accuracy',
        'calorie_head': 'mae'
    }
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)


## 9. Evaluation
Target metrik: Classification Accuracy ≥ 85%, Calorie MAE ≤ 0.02


In [ ]:
print("Evaluating on test set...")
results = model.evaluate(test_ds)

for name, value in zip(model.metrics_names, results):
    print(f"{name}: {value:.4f}")

class_acc_idx = model.metrics_names.index('classification_head_accuracy')
cal_mae_idx = model.metrics_names.index('calorie_head_mae')

print("-" * 30)
print(f"Classification Accuracy: {results[class_acc_idx]*100:.2f}% (Target: >= 85%)")
print(f"Calorie MAE (Normalized): {results[cal_mae_idx]:.4f} (Target: <= 0.02)")


## 10. Export Model


In [ ]:
export_dir = os.path.join(MODELS_DIR, 'vision_model')
model.save(export_dir)
print(f"SavedModel exported to {export_dir}")

converter = tf.lite.TFLiteConverter.from_saved_model(export_dir)
# converter.optimizations = [tf.lite.Optimize.DEFAULT] # Uncomment for quantization
tflite_model = converter.convert()

tflite_path = os.path.join(MODELS_DIR, 'vision_model.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print(f"TFLite model exported to {tflite_path}")


## 11. Download dari Colab


In [ ]:
import shutil
from google.colab import files

zip_path = os.path.join(GDRIVE_ROOT, "vision_model.zip")
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', export_dir)
print(f"Zipped SavedModel to {zip_path}")

# Uncomment lines below to download directly to local machine
# files.download(zip_path)
# files.download(tflite_path)
